# Baseline Selection

In [1]:
import json
from collections import defaultdict
from pathlib import Path

from edu_eval.statistics.metrics import bootstrap_ci

In [2]:
PROJECT_ROOT = Path.cwd().parent
MODEL_DIR = PROJECT_ROOT / "results" / "knowledge" / "smoke_qwen"

records = []
for name in ("primary_sd_smp.jsonl", "retention_sma.jsonl"):
    with (MODEL_DIR / name).open("r", encoding="utf-8") as f:
        records += [json.loads(line) for line in f if line.strip()]

def _correct(r):
    p = r["predictions"]
    key = "mean_log_likelihood" if "mean_log_likelihood" in p else "official"
    return bool(p[key]["correct"])

flags = [1.0 if _correct(r) else 0.0 for r in records]
low, high = bootstrap_ci(flags, seed=42)
print(f"overall: n={len(flags)} acc={sum(flags) / len(flags):.4f} [{low:.4f}, {high:.4f}]")
by_level: dict[str, list[float]] = defaultdict(list)
for r, v in zip(records, flags):
    by_level[r["level"]].append(v)
for level in sorted(by_level):
    vals = by_level[level]
    l, h = bootstrap_ci(vals, seed=42)
    print(f"{level}: n={len(vals)} acc={sum(vals) / len(vals):.4f} [{l:.4f}, {h:.4f}]")

overall: n=12860 acc=0.3083 [0.3003, 0.3162]


SD: n=4520 acc=0.3456 [0.3321, 0.3595]


SMA: n=4742 acc=0.2682 [0.2554, 0.2811]


SMP: n=3598 acc=0.3143 [0.2996, 0.3296]


In [3]:
print("G1 knowledge: SMOKE DATA")
print("G2 grade alignment: PENDING")
print("G3 correctness: PENDING")
print("G4 SMA retention: SMOKE DATA")
print("Pareto: PENDING")

G1 knowledge: SMOKE DATA
G2 grade alignment: PENDING
G3 correctness: PENDING
G4 SMA retention: SMOKE DATA
Pareto: PENDING
